# Assessment 2: Machine Learning and Real-Time Streaming

**Student ID:** 35415169
**Unit:** ITO5202 – Data Processing for Big Data

## Part A: Machine Learning Pipeline

This assessment reuses the airline on-time performance dataset from Assessment 1. The dataset will first be divided into a historical training subset and a fully held-out streaming subset so that records replayed through Kafka in Part B have not been used during model training.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Assessment2_ML_Streaming")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.5


In [4]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
) 

airline_schema = StructType([
    StructField("YEAR", IntegerType(), True),
    StructField("QUARTER", IntegerType(), True),
    StructField("MONTH", IntegerType(), True),
    StructField("DAY_OF_MONTH", IntegerType(), True),
    StructField("DAY_OF_WEEK", IntegerType(), True),
    StructField("FL_DATE", StringType(), True),
    StructField("OP_UNIQUE_CARRIER", StringType(), True),
    StructField("TAIL_NUM", StringType(), True),
    StructField("OP_CARRIER_FL_NUM", IntegerType(), True),
    StructField("ORIGIN_AIRPORT_ID", IntegerType(), True),
    StructField("ORIGIN", StringType(), True),
    StructField("ORIGIN_CITY_NAME", StringType(), True),
    StructField("ORIGIN_STATE_ABR", StringType(), True),
    StructField("DEST_AIRPORT_ID", IntegerType(), True),
    StructField("DEST", StringType(), True),
    StructField("DEST_CITY_NAME", StringType(), True),
    StructField("DEST_STATE_ABR", StringType(), True),
    StructField("CRS_DEP_TIME", IntegerType(), True),
    StructField("DEP_TIME", IntegerType(), True),
    StructField("DEP_DELAY", DoubleType(), True),
    StructField("DEP_DELAY_NEW", DoubleType(), True),
    StructField("DEP_DEL15", DoubleType(), True),
    StructField("TAXI_OUT", DoubleType(), True),
    StructField("TAXI_IN", DoubleType(), True),
    StructField("CRS_ARR_TIME", IntegerType(), True),
    StructField("ARR_TIME", IntegerType(), True),
    StructField("ARR_DELAY", DoubleType(), True),
    StructField("ARR_DELAY_NEW", DoubleType(), True),
    StructField("ARR_DEL15", DoubleType(), True),
    StructField("CANCELLED", DoubleType(), True),
    StructField("DIVERTED", DoubleType(), True),
    StructField("CRS_ELAPSED_TIME", DoubleType(), True),
    StructField("ACTUAL_ELAPSED_TIME", DoubleType(), True),
    StructField("AIR_TIME", DoubleType(), True),
    StructField("FLIGHTS", DoubleType(), True),
    StructField("DISTANCE", DoubleType(), True)
])
    

In [5]:
file_path = "../T_ONTIME_REPORTING.csv"


df = (
    spark.read
    .option("header", True)
    .schema(airline_schema)
    .csv(file_path)
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 539747
Number of columns: 36


### Target preparation

The machine learning task is binary classification using 'ARR_DEL15' as the target variable. Records with a missing target are excluded because cancelled or diverted flights may not have a valid arrival-delay outcome.

In [6]:
ml_df = df.filter(F.col("ARR_DEL15").isNotNull())
print("Rows with valid ARR_DEL15 target:", ml_df.count())

Rows with valid ARR_DEL15 target: 522269


In [7]:
ml_df.groupBy("ARR_DEL15").count().orderBy("ARR_DEL15").show()

+---------+------+
|ARR_DEL15| count|
+---------+------+
|      0.0|424139|
|      1.0| 98130|
+---------+------+



### Train/stream split

A reproducible 70/30 split is applied using a fixed random seed of 42. The 70% subset is reserved for machine learning model development and evaluation, while the remaining 30% is held out entirely for the simulated real-time streaming stage in Part B.

In [8]:
train_df, stream_df = ml_df.randomSplit([0.7, 0.3], seed=42)

train_count = train_df.count()
stream_count = stream_df.count()

print("Training subset:", train_count)
print("Streaming subset:", stream_count)
print("Total:", train_count + stream_count)

Training subset: 365710
Streaming subset: 156559
Total: 522269


In [9]:
total_count = train_count + stream_count

train_pct = train_count / total_count * 100
stream_pct = stream_count / total_count * 100

print("Training percentage:", round(train_pct, 2), "%")
print("Streaming percentage:", round(stream_pct, 2), "%")

Training percentage: 70.02 %
Streaming percentage: 29.98 %


In [10]:
stream_output_path = "data/stream_data.parquet"

stream_df.write.mode("overwrite").parquet(stream_output_path)

print("streaming subset saved to:", stream_output_path)

streaming subset saved to: data/stream_data.parquet


In [11]:
stream_check = spark.read.parquet(stream_output_path)

print("Reloaded streaming rows:", stream_check.count())

Reloaded streaming rows: 156559


The streaming subset is held out completely from model development so that the records replayed through Kafka in Part B represent unseen data. This avoids information leakage between the historical training phase and the simulated production inference phase.

## Feature Selection and Leakage Prevention

The target variable for the classification task is 'ARR_DEL15', which indicates whether a flight arrived at least 15 minutes late. Only features that would be available at, or shortly after, departure are considered for training. Variables that directly describe the final arrival outcome are excluded to avoid data leakage.

Candidate predictors include:

- 'MONTH'
- 'DAY_OF_WEEK'
- 'OP_UNIQUE_CARRIER'
- 'ORIGIN'
- 'DEST'
- 'CRS_DEP_TIME'
- 'CRS_ARR_TIME'
- 'DISTANCE'
- 'DEP_DELAY'

The following variables are excluded because they directly reveal, or are strongly derived from , the final arrival outcome:

- `ARR_DELAY`
- 'ARR_DELAY_NEW'
- 'ARR_DEL15'
- 'ARR_TIME'
- 'ACTUAL_ELAPSED_TIME'
- 'AIR_TIME'
- 'TAXI_IN'

This helps ensure that the model learns from information that would realistically be available before the final arrival outcome is known.

In [12]:
categorical_features = [
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST"
]


numeric_features = [
    "MONTH",
    "DAY_OF_WEEK",
    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "DISTANCE",
    "DEP_DELAY"
]

target_col = "ARR_DEL15"

selected_columns = categorical_features + numeric_features + [target_col]

ml_selected_df = train_df.select(*selected_columns)

ml_selected_df.show(5, truncate=False)

+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|MONTH|DAY_OF_WEEK|CRS_DEP_TIME|CRS_ARR_TIME|DISTANCE|DEP_DELAY|ARR_DEL15|
+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
|AA               |JFK   |SFO |1    |3          |600         |940         |2586.0  |-3.0     |0.0      |
|AA               |SFO   |JFK |1    |3          |1030        |1912        |2586.0  |-6.0     |0.0      |
|AA               |LAX   |JFK |1    |3          |1130        |2001        |2475.0  |41.0     |0.0      |
|AA               |JFK   |LAX |1    |3          |2100        |29          |2475.0  |-8.0     |0.0      |
|AA               |BOS   |LAX |1    |3          |801         |1140        |2611.0  |-5.0     |0.0      |
+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
only showing top 5 rows



In [13]:
missing_counts = ml_selected_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ml_selected_df.columns
])

missing_counts.show(truncate=False)

+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|MONTH|DAY_OF_WEEK|CRS_DEP_TIME|CRS_ARR_TIME|DISTANCE|DEP_DELAY|ARR_DEL15|
+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+
|0                |0     |0   |0    |0          |0           |0           |0       |0        |0        |
+-----------------+------+----+-----+-----------+------------+------------+--------+---------+---------+



No missing values were found in the selected modelling columns, so no imputation stage was required. Avoiding unnecessary imputation keeps the preprocessing pipeline simpler and prevents introducing artificial values into the feature space.

### Feature Engineering Pipeline

Categorical variables are converted to numerical representations using 'StringIndexer' followed by 'OneHotEncoder'. The encoded categorical variables and numeric predictors are then combined using 'VectorAssembler' into a single 'features' vector required by Spark ML models.

A null-value check confirmed that the selected modelling columns contain no missing values, so additional imputation is not required.



`DEP_DELAY` is retained because the intended prediction point is at or shortly after departure, when departure-delay information would already be available operationally.

In [14]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_INDEX",
        handleInvalid="keep"
    )
    for col in categorical_features
]

encoder = OneHotEncoder(
    inputCols=[f"{col}_INDEX" for col in categorical_features],
        outputCols=[f"{col}_OHE" for col in categorical_features]
)

assembler = VectorAssembler(
    inputCols=[
        *[f"{col}_OHE" for col in categorical_features],
        *numeric_features
    ],
    outputCol="features"
    
)

preprocessing_pipeline = Pipeline(
    stages=indexers + [encoder, assembler]
    
)
    



In [15]:
preprocessing_model = preprocessing_pipeline.fit(ml_selected_df)

transformed_df = preprocessing_model.transform(ml_selected_df)

transformed_df.select(
    "OP_UNIQUE_CARRIER",
    "ORIGIN",
    "DEST",
    "features",
    "ARR_DEL15"
).show(5,truncate=False)

+-----------------+------+----+----------------------------------------------------------------------------------------+---------+
|OP_UNIQUE_CARRIER|ORIGIN|DEST|features                                                                                |ARR_DEL15|
+-----------------+------+----+----------------------------------------------------------------------------------------+---------+
|AA               |JFK   |SFO |(678,[2,34,355,672,673,674,675,676,677],[1.0,1.0,1.0,1.0,3.0,600.0,940.0,2586.0,-3.0])  |0.0      |
|AA               |SFO   |JFK |(678,[2,24,363,672,673,674,675,676,677],[1.0,1.0,1.0,1.0,3.0,1030.0,1912.0,2586.0,-6.0])|0.0      |
|AA               |LAX   |JFK |(678,[2,21,363,672,673,674,675,676,677],[1.0,1.0,1.0,1.0,3.0,1130.0,2001.0,2475.0,41.0])|0.0      |
|AA               |JFK   |LAX |(678,[2,34,349,672,673,674,675,676,677],[1.0,1.0,1.0,1.0,3.0,2100.0,29.0,2475.0,-8.0])  |0.0      |
|AA               |BOS   |LAX |(678,[2,27,349,672,673,674,675,676,677],[1.0,1.0,1.0

## Model Development and Evaluation

The historical machine-learning subset is further divided into model traning and testing data. The training portion is used to fit candidate models, while the test portion is reserved for evaluation.

A fixed random seed is used so that the split can be reproduced.

In [16]:
model_train_df, model_test_df = ml_selected_df.randomSplit(
    [0.8, 0.2],
    seed=42
    
)

print("Model training rows:", model_train_df.count())
print("Model testing rows:", model_test_df.count())

Model training rows: 292799
Model testing rows: 72911


### Model 1: Logistic Regression

Logistic Regression is used as a baseline binary classification model for predicting whether a flight will arrive at least 15 minutes late. It is suitable for this task because the target variable is binary and the model provides a clear probabilistic classification output.

In [17]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="ARR_DEL15",
    predictionCol="prediction",
    probabilityCol="probability"
)

lr_pipeline = Pipeline(
    stages=indexers + [encoder, assembler, lr]
)

lr_model = lr_pipeline.fit(model_train_df)

lr_predictions = lr_model.transform(model_test_df)

lr_predictions.select(
    "ARR_DEL15",
    "prediction",
    "probability"
).show(10, truncate=False)

+---------+----------+------------------------------------------+
|ARR_DEL15|prediction|probability                               |
+---------+----------+------------------------------------------+
|0.0      |0.0       |[0.9642636541370255,0.03573634586297447]  |
|0.0      |0.0       |[0.9585785986704232,0.041421401329576835] |
|0.0      |0.0       |[0.9646324050595622,0.03536759494043784]  |
|0.0      |0.0       |[0.9651424585936924,0.03485754140630759]  |
|1.0      |1.0       |[4.0403299279292095E-8,0.9999999595967007]|
|0.0      |0.0       |[0.9510541854688341,0.048945814531165865] |
|0.0      |0.0       |[0.9855633601901622,0.014436639809837781] |
|0.0      |0.0       |[0.9347663400991875,0.06523365990081254]  |
|0.0      |0.0       |[0.9451572410798553,0.054842758920144674] |
|1.0      |0.0       |[0.909615484080688,0.09038451591931196]   |
+---------+----------+------------------------------------------+
only showing top 10 rows



In [18]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# AUC-ROC
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="ARR_DEL15",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

lr_auc = auc_evaluator.evaluate(lr_predictions)

# confusion matrix values
tp = lr_predictions.filter(
    (F.col("ARR_DEL15") == 1.0) & (F.col("prediction") == 1.0)
).count()

tn = lr_predictions.filter(
    (F.col("ARR_DEL15") == 0.0) & (F.col("prediction") == 0.0)
).count()

fp = lr_predictions.filter(
    (F.col("ARR_DEL15") == 0.0) & (F.col("prediction") == 1.0)
).count()

fn = lr_predictions.filter(
    (F.col("ARR_DEL15") == 1.0) & (F.col("prediction") == 0.0)
).count()

accuracy = (tp + tn) / (tp + tn + fp+ fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = (
    2 * precision * recall/ (precision + recall)
    if (precision + recall) > 0
    else 0
)

print("Logistic Regression")
print("AUC-ROC:", round(lr_auc, 4))
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

print("\nConfusion Matrix Counts")
print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)


Logistic Regression
AUC-ROC: 0.9257
Accuracy: 0.9248
Precision: 0.8829
Recall: 0.6908
F1-score: 0.7751

Confusion Matrix Counts
TP: 9452
TN: 57975
FP: 1254
FN: 4230


### Logistic Regression Results

The Logistic Regression model achieved an AUC-ROC of 0.9257 and an accuracy of 0.9248, indicating strong overall classification performance. Precision for the delayed-flight class was 0.8829, while recall was lower at 0.6908. This mean that most flights predicted as delayed were correctly identified, but the model failed to detect some actual delayed flights.

Because the target classes are imbalanced, accuracy alone is not sufficient for evaluation. The F1-score of 0.7751 provides a more balanced measure of precision and recall and will be used together with AUC-ROC when comparing this model with the next candidate model.



### Model 2: Random Forest

Random Forest is used as the second candidate classification model. It combines multiple decision trees and can capture non-linear relationships and interactions between predictors. Its performance will be compared with Logistic Regression using the same held-out test dataset and the same evaluation metrics.

In [19]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
featuresCol="features",
labelCol="ARR_DEL15",
predictionCol="prediction",
probabilityCol="probability",
numTrees=50,
seed=42

)

rf_pipeline = Pipeline(
    stages=indexers + [encoder, assembler, rf]
)

rf_model = rf_pipeline.fit(model_train_df)

rf_predictions = rf_model.transform(model_test_df)

rf_predictions.select(
    "ARR_DEL15",
    "prediction",
    "probability"
).show(10, truncate=False)

+---------+----------+----------------------------------------+
|ARR_DEL15|prediction|probability                             |
+---------+----------+----------------------------------------+
|0.0      |0.0       |[0.832605286892902,0.16739471310709794] |
|0.0      |0.0       |[0.832605286892902,0.16739471310709794] |
|0.0      |0.0       |[0.832605286892902,0.16739471310709794] |
|0.0      |0.0       |[0.8215203507913347,0.17847964920866535]|
|1.0      |0.0       |[0.7019146654348033,0.29808533456519665]|
|0.0      |0.0       |[0.832605286892902,0.16739471310709794] |
|0.0      |0.0       |[0.8215203507913347,0.17847964920866535]|
|0.0      |0.0       |[0.8317506485685868,0.16824935143141329]|
|0.0      |0.0       |[0.825123803867332,0.17487619613266803] |
|1.0      |0.0       |[0.8208497566002209,0.17915024339977914]|
+---------+----------+----------------------------------------+
only showing top 10 rows



In [20]:
rf_auc = auc_evaluator.evaluate(rf_predictions)

rf_tp = rf_predictions.filter(
    (F.col("ARR_DEL15") == 1.0) & (F.col("prediction") == 1.0)
).count()

rf_tn = rf_predictions.filter(
    (F.col("ARR_DEL15") ==0.0) & (F.col("prediction") == 0.0)
).count()

rf_fp = rf_predictions.filter(
    (F.col("ARR_DEL15") ==0.0) & (F.col("prediction") == 1.0)
).count()

rf_fn = rf_predictions.filter(
    (F.col("ARR_DEL15") ==1.0) & (F.col("prediction") ==0.0)
).count()

rf_accuracy = (rf_tp + rf_tn) / (rf_tp + rf_tn + rf_fp + rf_fn)

rf_precision = (
    rf_tp / (rf_tp + rf_fp)
    if (rf_tp + rf_fp) > 0
    else 0
)

rf_recall = (
    rf_tp / (rf_tp + rf_fn)
    if (rf_tp + rf_fn) > 0
    else 0
    
)

rf_f1 = (
    2* rf_precision * rf_recall / (rf_precision + rf_recall)
    if (rf_precision + rf_recall) > 0
    else 0
)

print("Random Forest")
print("AUC-ROC:", round(rf_auc, 4))
print("Accuracy:", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall:", round(rf_recall, 4))
print("F1-score:", round(rf_f1, 4))

print("\nConfusion Matrix Counts")
print("TP:",rf_tp)
print("TN:",rf_tn)
print("FP:",rf_fp)
print("FN:",rf_fn)



Random Forest
AUC-ROC: 0.8333
Accuracy: 0.8123
Precision: 0
Recall: 0.0
F1-score: 0

Confusion Matrix Counts
TP: 0
TN: 59229
FP: 0
FN: 13682


### Random Forest Results

The Random Forest model achieved an AUC-ROC of 0.8333 and an accuracy of 0.8123. However, precision, recall, and F1-score for the delayed-flight class were all 0. The confusion matrix shows that the model predicted every test record as the majority class, with 59,229 true negatives and 13,682 false negatives, but no true positives.

This demonstrates the effect of class imbalance on model evaluation. Although the accuracy appears relatively high, the model is not useful for the business objective because it fails to identify any delayed flights. Therefore, accuracy alone is not an appropriate model-selection metric for this dataset.

In [21]:
comparison_data = [
    (
        "Logistic Regression",
        float(lr_auc),
        float(accuracy),
        float(precision),
        float(recall),
        float(f1)
    ),
    (
        "Random Forest",
        float(rf_auc),
        float(rf_accuracy),
        float(rf_precision),
        float(rf_recall),
        float(rf_f1)
    )
]

comparison_df = spark.createDataFrame(
    comparison_data,
["Model", "AUC_ROC", "Accuracy", "Precision", "Recall", "F1_Score"]
)

comparison_df.show(truncate=False)

+-------------------+------------------+------------------+------------------+------------------+------------------+
|Model              |AUC_ROC           |Accuracy          |Precision         |Recall            |F1_Score          |
+-------------------+------------------+------------------+------------------+------------------+------------------+
|Logistic Regression|0.9256590218957667|0.9247850118637791|0.8828694190173735|0.6908346732933781|0.7751353124487452|
|Random Forest      |0.8332727407291873|0.8123465595040529|0.0               |0.0               |0.0               |
+-------------------+------------------+------------------+------------------+------------------+------------------+



### Model Comparison and Selection

Logistic Regression clearly outperformed Random Forest for the delayed-flight classification task. Logistic Regression achieved substantially higher AUC-ROC, accuracy, precision, recall, and F1-score. In contrast, the Random Forest model predicted only the majority class and to failed to identify any delayed flights, resulting in zero precision, recall, and F1-score for delayed-flight class.


Because delayed flights represent the minority class, accuracy alone is not sufficient for model selection. The Logistic Regression model provides a much better balance between identifying delayed flights and limiting false positive predictions. Based on the evaluation results, Logistic Regression is selected as the final model for persistence and use in the streaming phase.

### Model Persistence and Verification

Logistic Regression was selected as the final classification model based on its superior evaluation results. The complete fitted spark ML PipelineModel, including feature preprocessing and classification stages, is saved for reuse in Part B. 

Persisting the full pipeline ensures that incoming streaming records undergo the same feature transformations as the historical traning data. The saved model will be reloaded and tested before streaming integration. 

In [23]:
from pyspark.ml import PipelineModel

# Save the complete fitted Logistic Regression pipeline
model_path = "models/a2_model"

lr_model.write().overwrite().save(model_path)

print("Model successfully saved to:", model_path)


Model successfully saved to: models/a2_model


In [24]:
from pyspark.ml import PipelineModel

# Reload the saved model
loaded_model = PipelineModel.load(model_path)

# Test predictions using unseen model-test records
sample_predictions = loaded_model.transform(model_test_df.limit(5))

sample_predictions.select(
    "ARR_DEL15",
    "prediction",
    "probability"
).show(truncate=False)

# Verify the prediction columns exist
print("Prediction column available:", "prediction" in sample_predictions.columns)
print("Probability column available:", "probability" in sample_predictions.columns)

+---------+----------+------------------------------------------+
|ARR_DEL15|prediction|probability                               |
+---------+----------+------------------------------------------+
|0.0      |0.0       |[0.9642636541370255,0.03573634586297447]  |
|0.0      |0.0       |[0.9585785986704232,0.041421401329576835] |
|0.0      |0.0       |[0.9646324050595622,0.03536759494043784]  |
|0.0      |0.0       |[0.9651424585936924,0.03485754140630759]  |
|1.0      |1.0       |[4.0403299279292095E-8,0.9999999595967007]|
+---------+----------+------------------------------------------+

Prediction column available: True
Probability column available: True


### Model Persistance Verification Results

The fitted Logistic Regression PipelineModel was successfully saved to models/a2_model and reloaded using Spark ML's persistence API. The reloaded model generated predictions for five held-out test records without retraining. Both the prediction and probability columns were confirmed to exist in the output schema. 

This verifies that the persisted model includes the required preprocessing and classification stages and can be reused during the Structured Streaming phase in Part B.


### Part B: Structured Streaming


In [1]:
import os

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.5 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Assessment2_ML_Streaming")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.5


In [2]:
kafka_test = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "bigdata-kafka:9092")
    .option("subscribe", "events")
    .load()
)

print("Kafka connector loaded successfully")
kafka_test.printSchema()

Kafka connector loaded successfully
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



### Kafka Structured Streaming Consumer

The Spark Structured Streaming consumer reads simulated airline records from the kafka 'events' topic. Incoming Kafka messages contain JSON  arrays of flight records. These messages will be parsed using an explicitly declared schema and expanded into individual records before applying the persisted Logistic Regression model.

In [3]:
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "bigdata-kafka:9092")
    .option("subscribe", "events")
    .option("startingOffsets", "earliest")
    .load()
)

print("Kafka streaming source created successfully")
kafka_stream.printSchema()

Kafka streaming source created successfully
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



### JSON Parsing and Schema Declaration

The Kafka producer publishes each batch as a JSON array. An explicit schema is used to parse the incoming records, ensuring consistent data types and avoiding schema inference during streaming. Each array is expanded into individual flight records for subsequent model prediction.

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.sql.functions import col, from_json, explode, to_timestamp

# Declare the schema before starting the stream
stream_schema = StructType(
    spark.read.parquet("data/stream_data.parquet").schema.fields
    + [StructField("event_timestamp", StringType(), True)]
)

# Convert Kafka JSON arrays into individual flight records
parsed_stream = (
    kafka_stream
    .selectExpr("CAST(value AS STRING) AS json_value")
    .select(
        from_json(
            col("json_value"),
            ArrayType(stream_schema)
        ).alias("records")
    )
    .select(explode(col("records")).alias("record"))
    .select("record.*")
    .withColumn(
        "event_timestamp",
        to_timestamp(col("event_timestamp"))
    )
)

parsed_stream.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
